In [13]:
import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)


In [14]:
from analyse.advertising.e01_rupture_detection.script import (
    RuptureDetector,
    print_summary,
)
from dateutil import tz
from datetime import datetime
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)

tz_paris = tz.gettz("Europe/Paris")
start_time = datetime(2025, 3, 4, 13, 00, 0, tzinfo=tz_paris)
end_time = datetime(2025, 3, 4, 14, 0, 0, tzinfo=tz_paris)

TESTIMONY_CHANNEL = "TF1"

# This file was saved with utc datetime:
INPUT_FILE = "/root/Workspace/quotaclimat/analyse/advertising/exports/tf1_2025-03-04_12-00-00_2025-03-04_13-00-00.mp3"
VIDEO_INPUT = (
    "https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4"
)

OUT_JSON = "segments.json"
OUT_PLOT = "rupture_analyse.png"

annotations = get_testimony_data(
    channel=TESTIMONY_CHANNEL,
    from_date=start_time,
    to_date=end_time,
    source_file="manual_pub_tagging.csv",
)

detector = RuptureDetector(
    sensitivity=0.1,
    context_sec=1.0,  # durée analysée de chaque côté d'un point pour évaluer la rupture.
    max_ruptures=0,  # 0 pour pas de limite
    sr=22050,  # Fréquence d'échantillonnage de travail
    hop_length=512,  # Pas entre frames (~23ms à 22050Hz)
    n_mfcc=10,  # Nombre de coefficients MFCC
    novelty_smooth_sec=0.5,  # Lissage temporel de la courbe de nouveauté.
    min_segment_sec=1.0,  # Durée minimale d'un segment pour être considéré comme tel.
    silence_percentile=8.0,  # Percentile pour définir le seuil de silence
    cosine_weight=1.0,
)

In [15]:
from analyse.advertising.e01_rupture_detection.script import export_json


segments, novelty, features, y = detector.run(INPUT_FILE)

print_summary(segments)

export_json(segments, "segments.json")

generate_player(
    media_input="https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp3",
    segments=[s.to_dict() for s in segments],
    annotations=format_annotation(annotations, from_date=start_time),
    output_path="output_cosine.html",
    params=detector.get_params(),
    novelty_peaks=detector.get_novelty_peaks(novelty),
    start_epoch=int(start_time.timestamp()),
)

[1/5] Chargement : /root/Workspace/quotaclimat/analyse/advertising/exports/tf1_2025-03-04_12-00-00_2025-03-04_13-00-00.mp3


      Durée : 3600.0s  (60.0 min)
[2/5] Extraction des features...
[3/5] Calcul de la courbe de nouveauté...
      Fenêtre de contexte : 43 frames = 1.00s de chaque côté
      Seuil silence       : énergie < 0.00458 (percentile 8.0) → 23705 frames silencieuses (15.3%)
      Lissage             : 21 frames = 0.49s
[4/5] Détection des frontières...
      Seuil sensitivity=0.10 → hauteur min 0.0064 (percentile 90)
      Après seuil + distance min (1.0s) : 571 ruptures candidates
      Résultat final : 571 ruptures → 572 segments
[5/5] Segmentation terminée : 572 segments

═══════════════════════════════════════════════════════
  RÉSUMÉ DE SEGMENTATION
═══════════════════════════════════════════════════════
  Total segments : 572
  Label                     Nb       %   Durée moy
  ──────────────────────────────────────────────────
  inconnu                  418   73.1%       3.4s
  pub_candidat             129   22.6%      16.5s
  silence                   25    4.4%       1.6s
══════════